## Gold — Modelagem

Neste notebook eu monto o modelo dimensional (esquema estrela com 3 fatos e 4 dimensões conformadas) a partir das tabelas da Silver. Cada tabela é criada com tipos, descrições (`COMMENT`) e chaves declaradas, para que o catálogo de dados fique registrado no próprio Unity Catalog.

### 1. Schema Gold

Crio o schema `gold` no catálogo `susep_capitalizacao`, que vai receber as dimensões e as fatos.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS susep_capitalizacao.gold

### 2. Dimensão `dim_empresa`

Crio a dimensão de empresas a partir da `silver.empresas`, declarando tipos, descrições (`COMMENT`) e a chave primária. Em seguida, carrego os dados com `INSERT`. É a dimensão compartilhada pelas 3 fatos.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.dim_empresa (
  coenti       STRING NOT NULL COMMENT 'Código da empresa na SUSEP (PK). 5 dígitos, com zeros à esquerda. Origem: Ses_cias.Coenti (trim)',
  nome_empresa STRING NOT NULL COMMENT 'Razão social da empresa. Origem: Ses_cias.Noenti (trim)',
  CONSTRAINT pk_dim_empresa PRIMARY KEY (coenti)
)
COMMENT 'Companhias de capitalização com movimento nos arquivos da SUSEP em 2021–2025 (19 empresas). Origem: Ses_cias.csv → silver.empresas';

INSERT INTO susep_capitalizacao.gold.dim_empresa
SELECT coenti, nome_empresa
FROM susep_capitalizacao.silver.empresas;

SELECT COUNT(*) AS linhas FROM susep_capitalizacao.gold.dim_empresa;

### 3. Dimensão `dim_tempo`

Gero o calendário mensal do período analisado, de 01/2021 a 12/2025, com uma linha por mês. A chave `damesano` segue o formato `AAAAMM` das fontes da SUSEP, e as colunas derivadas (data, ano, mês e trimestre) facilitam os agrupamentos na análise. É compartilhada pelas 3 fatos.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.dim_tempo (
  damesano  INT  NOT NULL COMMENT 'Ano-mês no formato AAAAMM (PK). Domínio: 202101 a 202512',
  data_ref  DATE NOT NULL COMMENT 'Primeiro dia do mês. Domínio: 2021-01-01 a 2025-12-01. Derivado de damesano',
  ano       INT  NOT NULL COMMENT 'Ano. Domínio: 2021 a 2025',
  mes       INT  NOT NULL COMMENT 'Mês. Domínio: 1 a 12',
  trimestre INT  NOT NULL COMMENT 'Trimestre do ano. Domínio: 1 a 4',
  CONSTRAINT pk_dim_tempo PRIMARY KEY (damesano)
)
COMMENT 'Calendário mensal do período analisado (60 meses, 2021–2025). Gerado no pipeline, sem arquivo de origem';

INSERT INTO susep_capitalizacao.gold.dim_tempo
SELECT
  CAST(DATE_FORMAT(d, 'yyyyMM') AS INT) AS damesano,
  d                                     AS data_ref,
  YEAR(d)                               AS ano,
  MONTH(d)                              AS mes,
  QUARTER(d)                            AS trimestre
FROM (
  SELECT EXPLODE(SEQUENCE(DATE'2021-01-01', DATE'2025-12-01', INTERVAL 1 MONTH)) AS d
);

SELECT COUNT(*) AS linhas, MIN(damesano) AS primeiro, MAX(damesano) AS ultimo
FROM susep_capitalizacao.gold.dim_tempo;

### 4. Dimensão `dim_uf`

Crio a dimensão de Unidades Federativas a partir das siglas distintas da `silver.cap_uf`, enriquecidas com o nome da UF e a região geográfica conforme a divisão regional do IBGE. O mapeamento é fixo no código, porque a fonte da SUSEP traz só a sigla. A região permite analisar a concentração geográfica (pergunta 2) também por região.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.dim_uf (
  uf      STRING NOT NULL COMMENT 'Sigla da UF (PK). Domínio: 27 siglas (AC a TO). Origem: ses_cap_uf.UF (upper)',
  nome_uf STRING NOT NULL COMMENT 'Nome da UF. Origem: mapeamento IBGE fixo no pipeline',
  regiao  STRING NOT NULL COMMENT 'Região geográfica. Domínio: Norte, Nordeste, Centro-Oeste, Sudeste, Sul. Origem: mapeamento IBGE',
  CONSTRAINT pk_dim_uf PRIMARY KEY (uf)
)
COMMENT 'Unidades Federativas do Brasil (27). Origem: siglas distintas de ses_cap_uf.csv → silver.cap_uf + nome e região do IBGE';

INSERT INTO susep_capitalizacao.gold.dim_uf
SELECT d.uf, m.nome_uf, m.regiao
FROM (SELECT DISTINCT uf FROM susep_capitalizacao.silver.cap_uf) d
LEFT JOIN (VALUES
  ('AC','Acre','Norte'), ('AM','Amazonas','Norte'), ('AP','Amapá','Norte'), ('PA','Pará','Norte'),
  ('RO','Rondônia','Norte'), ('RR','Roraima','Norte'), ('TO','Tocantins','Norte'),
  ('AL','Alagoas','Nordeste'), ('BA','Bahia','Nordeste'), ('CE','Ceará','Nordeste'),
  ('MA','Maranhão','Nordeste'), ('PB','Paraíba','Nordeste'), ('PE','Pernambuco','Nordeste'),
  ('PI','Piauí','Nordeste'), ('RN','Rio Grande do Norte','Nordeste'), ('SE','Sergipe','Nordeste'),
  ('DF','Distrito Federal','Centro-Oeste'), ('GO','Goiás','Centro-Oeste'),
  ('MS','Mato Grosso do Sul','Centro-Oeste'), ('MT','Mato Grosso','Centro-Oeste'),
  ('ES','Espírito Santo','Sudeste'), ('MG','Minas Gerais','Sudeste'),
  ('RJ','Rio de Janeiro','Sudeste'), ('SP','São Paulo','Sudeste'),
  ('PR','Paraná','Sul'), ('RS','Rio Grande do Sul','Sul'), ('SC','Santa Catarina','Sul')
) AS m(uf, nome_uf, regiao)
  ON d.uf = m.uf;

SELECT regiao, COUNT(*) AS ufs
FROM susep_capitalizacao.gold.dim_uf
GROUP BY regiao
ORDER BY regiao;

### 5. Dimensão `dim_modalidade`

Crio a dimensão de modalidades de título de capitalização a partir dos pares distintos de código e descrição da `silver.cap_modalidade`. O código 0, publicado pela SUSEP sem descrição, já chega da Silver como "Não informada". A modalidade 2 (Compra-Programada) não aparece no período 2021–2025, então a dimensão tem 7 modalidades.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.dim_modalidade (
  cod_modalidade INT    NOT NULL COMMENT 'Código da modalidade (PK). Domínio: 0, 1, 3, 4, 5, 6, 7 (o código 2 não ocorre em 2021–2025). Origem: Ses_Dados_Cap.codModal',
  modalidade     STRING NOT NULL COMMENT 'Descrição da modalidade. Origem: Ses_Dados_Cap.modalidade; código 0 (sem descrição na origem) → "Não informada"',
  CONSTRAINT pk_dim_modalidade PRIMARY KEY (cod_modalidade)
)
COMMENT 'Modalidades de título de capitalização (7 no período). Origem: pares distintos de Ses_Dados_Cap.csv → silver.cap_modalidade';

INSERT INTO susep_capitalizacao.gold.dim_modalidade
SELECT DISTINCT cod_modalidade, modalidade
FROM susep_capitalizacao.silver.cap_modalidade;

SELECT * FROM susep_capitalizacao.gold.dim_modalidade ORDER BY cod_modalidade;

### 6. Fato `fato_premios_uf`

Crio a fato de movimento mensal por empresa, mês e UF a partir da `silver.cap_uf`. É um snapshot periódico mensal com medidas aditivas (somam em qualquer dimensão). A chave primária é composta (`coenti`, `damesano`, `uf`), e cada parte dela é chave estrangeira para a dimensão correspondente.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.fato_premios_uf (
  coenti          STRING        NOT NULL COMMENT 'Empresa (FK → dim_empresa). Origem: ses_cap_uf.COENTI',
  damesano        INT           NOT NULL COMMENT 'Mês de referência AAAAMM (FK → dim_tempo). Domínio: 202101 a 202512. Origem: ses_cap_uf.DAMESANO',
  uf              STRING        NOT NULL COMMENT 'UF (FK → dim_uf). Domínio: 27 siglas. Origem: ses_cap_uf.UF (upper)',
  premio          DECIMAL(18,2) NOT NULL COMMENT 'Prêmios do mês em R$, líquidos de devoluções e cancelamentos. Pode ser negativo quando as devoluções superam a arrecadação. Origem: ses_cap_uf.PREMIO',
  resgate_pago    DECIMAL(18,2) NOT NULL COMMENT 'Resgates pagos no mês em R$. Domínio: >= 0. Origem: ses_cap_uf.RESGPAGO (negativos → 0)',
  sorteio_pago    DECIMAL(18,2) NOT NULL COMMENT 'Sorteios pagos no mês em R$. Domínio: >= 0. Origem: ses_cap_uf.SORTPAGO (negativos → 0)',
  qtd_resgatantes BIGINT        NOT NULL COMMENT 'Quantidade de resgatantes no mês. Domínio: >= 0. Origem: ses_cap_uf.RESGATANTES (arredondado; negativos → 0)',
  qtd_sorteios    BIGINT        NOT NULL COMMENT 'Quantidade de sorteios no mês. Domínio: >= 0. Origem: ses_cap_uf.SORTEIOS (arredondado; negativos → 0)',
  CONSTRAINT pk_fato_premios_uf PRIMARY KEY (coenti, damesano, uf),
  CONSTRAINT fk_premios_uf_empresa FOREIGN KEY (coenti)   REFERENCES susep_capitalizacao.gold.dim_empresa (coenti),
  CONSTRAINT fk_premios_uf_tempo   FOREIGN KEY (damesano) REFERENCES susep_capitalizacao.gold.dim_tempo (damesano),
  CONSTRAINT fk_premios_uf_uf      FOREIGN KEY (uf)       REFERENCES susep_capitalizacao.gold.dim_uf (uf)
)
COMMENT 'Movimento mensal de títulos de capitalização por empresa e UF. Grão: empresa × mês × UF. Snapshot periódico mensal, medidas aditivas. Origem: ses_cap_uf.csv → silver.cap_uf';

INSERT INTO susep_capitalizacao.gold.fato_premios_uf
SELECT coenti, damesano, uf, premio, resgate_pago, sorteio_pago, qtd_resgatantes, qtd_sorteios
FROM susep_capitalizacao.silver.cap_uf;

SELECT COUNT(*) AS linhas FROM susep_capitalizacao.gold.fato_premios_uf;

### 7. Fato `fato_modalidade`

Crio a fato de movimento mensal por empresa, mês e modalidade a partir da `silver.cap_modalidade`. É um snapshot periódico mensal com medidas aditivas. A descrição da modalidade fica só na `dim_modalidade`; a fato guarda apenas o código, que é a chave estrangeira.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.fato_modalidade (
  coenti         STRING        NOT NULL COMMENT 'Empresa (FK para dim_empresa). Origem: Ses_Dados_Cap.coenti',
  damesano       INT           NOT NULL COMMENT 'Mês de referência AAAAMM (FK para dim_tempo). Domínio: 202101 a 202512. Origem: Ses_Dados_Cap.damesano',
  cod_modalidade INT           NOT NULL COMMENT 'Modalidade (FK para dim_modalidade). Domínio: 0, 1, 3, 4, 5, 6, 7. Origem: Ses_Dados_Cap.codModal',
  receitas       DECIMAL(18,2) NOT NULL COMMENT 'Receitas com títulos no mês em R$, líquidas de devoluções e cancelamentos. Pode ser negativa quando as devoluções superam a arrecadação. Origem: Ses_Dados_Cap.receitasCap',
  resgates       DECIMAL(18,2) NOT NULL COMMENT 'Resgates no mês em R$. Domínio: >= 0. Origem: Ses_Dados_Cap.valorResg',
  sorteios_pagos DECIMAL(18,2) NOT NULL COMMENT 'Sorteios pagos no mês em R$. Domínio: >= 0. Origem: Ses_Dados_Cap.sorteiosPagos',
  CONSTRAINT pk_fato_modalidade PRIMARY KEY (coenti, damesano, cod_modalidade),
  CONSTRAINT fk_modalidade_empresa    FOREIGN KEY (coenti)         REFERENCES susep_capitalizacao.gold.dim_empresa (coenti),
  CONSTRAINT fk_modalidade_tempo      FOREIGN KEY (damesano)       REFERENCES susep_capitalizacao.gold.dim_tempo (damesano),
  CONSTRAINT fk_modalidade_modalidade FOREIGN KEY (cod_modalidade) REFERENCES susep_capitalizacao.gold.dim_modalidade (cod_modalidade)
)
COMMENT 'Movimento mensal de títulos de capitalização por empresa e modalidade. Grão: empresa × mês × modalidade. Snapshot periódico mensal, medidas aditivas. Origem: Ses_Dados_Cap.csv → silver.cap_modalidade';

INSERT INTO susep_capitalizacao.gold.fato_modalidade
SELECT coenti, damesano, cod_modalidade, receitas, resgates, sorteios_pagos
FROM susep_capitalizacao.silver.cap_modalidade;

SELECT COUNT(*) AS linhas FROM susep_capitalizacao.gold.fato_modalidade;

### 8. Fato `fato_provisao`

Crio a fato com o saldo mensal da provisão total por empresa a partir da `silver.provisao`. É um snapshot periódico mensal com medida **semiaditiva**: o saldo pode ser somado entre empresas (total do mercado em um mês), mas não entre meses, porque isso contaria o mesmo saldo várias vezes.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.gold.fato_provisao (
  coenti         STRING        NOT NULL COMMENT 'Empresa (FK para dim_empresa). Origem: Ses_prov.coenti',
  damesano       INT           NOT NULL COMMENT 'Mês de referência AAAAMM (FK para dim_tempo). Domínio: 202101 a 202512. Origem: Ses_prov.damesano',
  provisao_total DECIMAL(18,2) NOT NULL COMMENT 'Saldo da provisão total no fim do mês em R$. Domínio: >= 0. Semiaditiva: soma entre empresas, mas não entre meses. Origem: Ses_prov.valor',
  CONSTRAINT pk_fato_provisao PRIMARY KEY (coenti, damesano),
  CONSTRAINT fk_provisao_empresa FOREIGN KEY (coenti)   REFERENCES susep_capitalizacao.gold.dim_empresa (coenti),
  CONSTRAINT fk_provisao_tempo   FOREIGN KEY (damesano) REFERENCES susep_capitalizacao.gold.dim_tempo (damesano)
)
COMMENT 'Saldo mensal da provisão total das empresas de capitalização. Grão: empresa × mês. Snapshot periódico mensal, medida semiaditiva. Origem: Ses_prov.csv (todos os mercados) → silver.provisao (só empresas de capitalização)';

INSERT INTO susep_capitalizacao.gold.fato_provisao
SELECT coenti, damesano, provisao_total
FROM susep_capitalizacao.silver.provisao;

SELECT COUNT(*) AS linhas FROM susep_capitalizacao.gold.fato_provisao;

### 9. Checagem de unicidade

Verifico se a chave primária de cada tabela da Gold é de fato única, comparando o total de linhas com o total de chaves distintas. No Databricks, a `PRIMARY KEY` é informativa e não impede duplicatas na gravação, então a checagem é o que garante o grão de cada tabela. A diferença deve ser zero.


In [0]:
%sql
SELECT 'dim_empresa' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT coenti) AS chaves_distintas
FROM susep_capitalizacao.gold.dim_empresa
UNION ALL
SELECT 'dim_tempo', COUNT(*), COUNT(DISTINCT damesano)
FROM susep_capitalizacao.gold.dim_tempo
UNION ALL
SELECT 'dim_uf', COUNT(*), COUNT(DISTINCT uf)
FROM susep_capitalizacao.gold.dim_uf
UNION ALL
SELECT 'dim_modalidade', COUNT(*), COUNT(DISTINCT cod_modalidade)
FROM susep_capitalizacao.gold.dim_modalidade
UNION ALL
SELECT 'fato_premios_uf', COUNT(*), COUNT(DISTINCT coenti, damesano, uf)
FROM susep_capitalizacao.gold.fato_premios_uf
UNION ALL
SELECT 'fato_modalidade', COUNT(*), COUNT(DISTINCT coenti, damesano, cod_modalidade)
FROM susep_capitalizacao.gold.fato_modalidade
UNION ALL
SELECT 'fato_provisao', COUNT(*), COUNT(DISTINCT coenti, damesano)
FROM susep_capitalizacao.gold.fato_provisao

### 10. Checagem de integridade referencial

Verifico se toda chave estrangeira das 3 fatos existe na dimensão correspondente, ou seja, se não há registros órfãos. Assim como a chave primária, a `FOREIGN KEY` é informativa no Databricks, então é esta checagem que garante que nenhuma linha se perde ao cruzar fatos e dimensões na análise. O resultado deve ser zero nas 8 relações do modelo.


In [0]:
%sql
SELECT 'fato_premios_uf' AS fato, 'dim_empresa' AS dimensao, COUNT(*) AS orfaos
FROM susep_capitalizacao.gold.fato_premios_uf f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_empresa d ON f.coenti = d.coenti
UNION ALL
SELECT 'fato_premios_uf', 'dim_tempo', COUNT(*)
FROM susep_capitalizacao.gold.fato_premios_uf f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_tempo d ON f.damesano = d.damesano
UNION ALL
SELECT 'fato_premios_uf', 'dim_uf', COUNT(*)
FROM susep_capitalizacao.gold.fato_premios_uf f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_uf d ON f.uf = d.uf
UNION ALL
SELECT 'fato_modalidade', 'dim_empresa', COUNT(*)
FROM susep_capitalizacao.gold.fato_modalidade f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_empresa d ON f.coenti = d.coenti
UNION ALL
SELECT 'fato_modalidade', 'dim_tempo', COUNT(*)
FROM susep_capitalizacao.gold.fato_modalidade f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_tempo d ON f.damesano = d.damesano
UNION ALL
SELECT 'fato_modalidade', 'dim_modalidade', COUNT(*)
FROM susep_capitalizacao.gold.fato_modalidade f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_modalidade d ON f.cod_modalidade = d.cod_modalidade
UNION ALL
SELECT 'fato_provisao', 'dim_empresa', COUNT(*)
FROM susep_capitalizacao.gold.fato_provisao f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_empresa d ON f.coenti = d.coenti
UNION ALL
SELECT 'fato_provisao', 'dim_tempo', COUNT(*)
FROM susep_capitalizacao.gold.fato_provisao f
LEFT ANTI JOIN susep_capitalizacao.gold.dim_tempo d ON f.damesano = d.damesano